In [8]:
!pip install pandas numpy scikit-learn shap matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

print("--- STEP 1: LOADING & ALIGNING DATA ---")
df = pd.read_csv('cleaned_real_estate_data.csv')

# Drop any row missing critical targets to force a perfect row match
df = df.dropna(subset=['Price_INR', 'Area_SqFt', 'Locality', 'City', 'BHK'])
df.drop_duplicates(inplace=True)

# Define X and y from the exact same clean dataframe at the same time
X = df.drop('Price_INR', axis=1)
y = df['Price_INR']
print(f"Verified Safe Alignment -> Rows in X: {X.shape[0]}, Rows in y: {y.shape[0]}")

# Precompute Market Insights for Page 9 of your Dashboard
market_insights = df.groupby('Locality').agg({
    'Price_INR': 'mean',
    'Infrastructure_Growth_Score': 'mean',
    'Crime_Index': 'mean'
}).reset_index()
market_insights.to_csv('market_insights.csv', index=False)

# Define columns GLOBALLY so all blocks can access them
categorical_cols = ['Property_Type', 'Furnished_Status', 'Locality', 'City']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Run the split inside the same execution block
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("✅ Data split completely successful!")

print("\n--- STEP 2: PREPROCESSING & TRAINING ---")
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training production Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_processed, y_train)
print(f"Model Training Complete! R² Accuracy Score: {r2_score(y_test, rf_model.predict(X_test_processed)):.4f}")

print("\n--- STEP 3: GENERATING EXPLAINABLE AI PLOTS (PAGE 7) ---")
# Reconstruct feature names safely using the global categorical and numeric arrays
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(encoded_cat_cols)

# Initialize and calculate SHAP tree explainer arrays
explainer = shap.TreeExplainer(rf_model)

# Sample 100 rows for high-speed calculation to avoid notebook timeouts
X_sample = X_test_processed[:100]
if hasattr(X_sample, "toarray"):
    X_sample = X_sample.toarray()

shap_values = explainer.shap_values(X_sample)

# Handle both legacy list-based returns and modern 3D array outputs from SHAP safely
if isinstance(shap_values, list):
    shap_matrix = shap_values[0]
elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    shap_matrix = shap_values[:, :, 0]
else:
    shap_matrix = shap_values

# Generate and save the final summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_matrix, X_sample, feature_names=all_feature_names, show=False)
plt.title("SHAP Feature Importance Analysis", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', bbox_inches='tight', dpi=300)
plt.close()
print("Saved 'shap_summary_plot.png' successfully!")

# Export pipeline package assets for your 9-page Streamlit Dashboard app
best_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', rf_model)])
joblib.dump(best_pipeline, 'real_estate_best_model.pkl')
joblib.dump(numeric_cols, 'numeric_columns_list.pkl')
print("\n🏆 SUCCESS: All model assets exported with zero errors!")

--- STEP 1: LOADING & ALIGNING DATA ---
Verified Safe Alignment -> Rows in X: 21538, Rows in y: 21538
✅ Data split completely successful!

--- STEP 2: PREPROCESSING & TRAINING ---
Training production Random Forest Regressor...
Model Training Complete! R² Accuracy Score: 0.6386

--- STEP 3: GENERATING EXPLAINABLE AI PLOTS (PAGE 7) ---


UFuncTypeError: Cannot cast ufunc 'isnan' input from dtype('O') to dtype('bool') with casting rule 'same_kind'